# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laraib175/Flyrank-week2/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My Lane as an ML Task

**Task Type:** Ranking / Scoring (supervised, but output is a ranked list)

I am framing this as a **ranking** problem rather than pure classification. The editor does not need a hard "declining / not declining" label — they need an ordered queue: "Which page should I review first?" The model's output is a continuous priority score, and the action is: work top-down until capacity runs out.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or Proxy

**Proxy:** A binary flag `is_opportunity` built from **observed future behavior**, not from an existing derived column.

Definition sketch: A page is a positive (1) if, in the 30 days **after** the feature window, its `gsc_clicks` drops by more than X% relative to the prior 30 days, **or** if its CTR in the feature window is below the 25th percentile for its position tier among pages with meaningful impressions.

**Why not `is_declining_label`:** That column is derived from `trend_pct`, which is in the same row as the features. Using it as the target means the model learns the label formula, not a real-world pattern. The target must come from a later time window than the features.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success Metric

**Metric:** Precision@50

**Why:** The editor's weekly review capacity is roughly 50 pages. The cost of a false positive is a wasted editor hour; the cost of a false negative is a missed page that could have been fixed. Precision@50 directly measures: among the 50 pages we told the editor to fix first, how many were actually worth fixing? Accuracy over 30,000 rows is the wrong question.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
!pwd
!ls -la

/content
total 16
drwxr-xr-x 1 root root 4096 Nov  6 16:29 .
drwxr-xr-x 1 root root 4096 Nov  7 09:08 ..
drwxr-xr-x 4 root root 4096 Nov  6 16:29 .config
drwxr-xr-x 1 root root 4096 Nov  6 16:29 sample_data


In [4]:
%cd /content
!rm -rf Flyrank-week2
!git clone https://github.com/Laraib175/Flyrank-week2.git
%cd Flyrank-week2
!ls -la
!find . -name "*.csv"

/content
Cloning into 'Flyrank-week2'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 119 (delta 31), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.84 MiB | 8.05 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/Flyrank-week2
total 104
drwxr-xr-x 12 root root  4096 Nov  7 09:19 .
drwxr-xr-x  1 root root  4096 Nov  7 09:19 ..
-rw-r--r--  1 root root   654 Nov  7 09:19 AGENTS.md
-rw-r--r--  1 root root   654 Nov  7 09:19 CLAUDE.md
drwxr-xr-x  3 root root  4096 Nov  7 09:19 data
-rw-r--r--  1 root root  2763 Nov  7 09:19 DATA_USE.md
drwxr-xr-x  3 root root  4096 Nov  7 09:19 docs
drwxr-xr-x  8 root root  4096 Nov  7 09:19 .git
drwxr-xr-x  3 root root  4096 Nov  7 09:19 .github
-rw-r--r--  1 root root   993 Nov  7 09:19 .gitignore
-rw-r--r--  1 root root 10408 Nov  7 09:19 GUIDE.md
-rw-r--r--  1 root root  1289 Nov  7 09:19 LICENSE
drwxr-xr

In [6]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [7]:
cols_to_show = ["content_id", "client_id", "content_type", "ctr",
                "avg_position", "impressions_90d", "clicks_90d"]
df[cols_to_show].head(10)

,content_id,client_id,content_type,ctr,avg_position,impressions_90d,clicks_90d
0,content_304f48230142,client_f369cb89fc,keyword article,0.76,10.6,3803,29
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0.05,20.3,15320,7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.09,36.5,12581,11
3,content_331d6c4de07b,client_19581e27de,keyword article,0.49,6.2,11751,58
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.13,44.0,19140,24
5,content_d4084a4bc775,client_f369cb89fc,keyword article,0.03,8.5,3970,1
6,content_9a34b442b552,client_8722616204,keyword article,0.00,7.0,20,0
7,content_a63219c6e95a,client_19581e27de,keyword article,0.06,21.2,1724,1
8,content_5e6c160719bc,client_6208ef0f77,keyword article,0.09,46.0,32574,29
9,content_c27558df2b0c,client_19581e27de,keyword article,0.16,4.9,1240,2


In [8]:
!ls work/notebooks/


capstone.ipynb			 w04_baseline_score.ipynb
w01_research_question.ipynb	 w04_signal_audit.ipynb
w02_ml_task_framing.ipynb	 w05_model.ipynb
w03_data_contract.ipynb		 w06_validation_audit.ipynb
w03_feature_leakage_check.ipynb  w07_action_playbook.ipynb


In [9]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")

assert df["content_id"].is_unique, "content_id is not unique!"
print("content_id is unique. One row = one content item.")

# build the proxy target
df["is_declining_future"] = (df["clicks_last_30d"] < df["clicks_prev_30d"] * 0.8).astype(int)
print("positive rate:", df["is_declining_future"].mean().round(3))

cols_to_show = ["content_id", "client_id", "content_type", "ctr",
                "avg_position", "impressions_90d", "clicks_90d",
                "clicks_prev_30d", "clicks_last_30d", "is_declining_future"]
df[cols_to_show].head(10)

Rows: 30000, Columns: 44
content_id is unique. One row = one content item.
positive rate: 0.201


,content_id,client_id,content_type,ctr,avg_position,impressions_90d,clicks_90d,clicks_prev_30d,clicks_last_30d,is_declining_future
0,content_304f48230142,client_f369cb89fc,keyword article,0.76,10.6,3803,29,13,2,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0.05,20.3,15320,7,1,2,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.09,36.5,12581,11,3,1,1
3,content_331d6c4de07b,client_19581e27de,keyword article,0.49,6.2,11751,58,17,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.13,44.0,19140,24,2,10,0
5,content_d4084a4bc775,client_f369cb89fc,keyword article,0.03,8.5,3970,1,1,0,1
6,content_9a34b442b552,client_8722616204,keyword article,0.00,7.0,20,0,0,0,0
7,content_a63219c6e95a,client_19581e27de,keyword article,0.06,21.2,1724,1,0,1,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,0.09,46.0,32574,29,8,9,0
9,content_c27558df2b0c,client_19581e27de,keyword article,0.16,4.9,1240,2,0,0,0


## 4. Unit of Analysis

**One row = one pseudonymized content page.** Verified via `content_id.is_unique` returning True.

This matters because: if I later aggregate to client level or query level, the meaning of the score changes. The editor acts on **pages**, so the unit must be the page.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML Beats a Fixed Rule

**A plain rule is not enough because:**

1. **A flat CTR threshold is too blunt.** A page ranking #2 with 0.3% CTR is a disaster; a page ranking #40 with 0.3% CTR may be normal. Position tier must be adjusted for, and the adjustment factor is not a single constant.

2. **Signals are tangled.** CTR, engagement, position, content type, and traffic volume interact non-linearly. A hand-written if-statement either misses opportunities or floods the queue.

3. **The cost structure demands ranking, not binary flagging.** A rule can only say "flag/not flag". The editor needs "first / second / third...". A scoring model produces a continuous ordering that a threshold rule cannot.

**What I will claim:** decision-support ranking, not causal claims. A high score means "worth reviewing first", not "fixing this will increase traffic".

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.